# 🔀 EidCloud Workflow Engine - Quickstart

Welcome to **eidcloud-workflow**, a lightweight, production-ready, declarative JSON/YAML workflow automation engine written in pure PHP 8.2+ with zero external vendor dependencies.

This quickstart demonstrates:
1. Setting up PHP and cloning the repository.
2. Defining a multi-step declarative workflow with conditional branching and output interpolation.
3. Executing the workflow engine programmatically and inspecting checkpoints.

In [ ]:
# [Cell 1] Environment Setup & Verification
!apt-get update -qq && apt-get install -y php-cli php-curl php-sqlite3 -qq
!php -v

# Clone or link repository if not already in directory
import os
if not os.path.exists('eidcloud-workflow'):
    !git clone https://github.com/eidcloud/eidcloud-workflow.git
    %cd eidcloud-workflow
else:
    %cd eidcloud-workflow

# Run automated test suite verifying zero-dependency engine
!php tests/run_tests.php

In [ ]:
# [Cell 2] Define Declarative Workflow (JSON)
workflow_definition = """{
  "id": "user_onboarding_demo",
  "name": "Colab User Onboarding & AI Enrichment",
  "steps": [
    {
      "id": "user_profile",
      "type": "transform",
      "config": {
        "operation": "map",
        "mapping": {
          "userId": "{{input.userId ?? 101}}",
          "name": "{{input.name ?? 'Alex Mercer'}}",
          "role": "{{input.role ?? 'developer'}}"
        }
      }
    },
    {
      "id": "check_role",
      "type": "condition",
      "config": {
        "condition": {
          "field": "{{steps.user_profile.role}}",
          "operator": "==",
          "value": "developer"
        },
        "then": "grant_sandbox_access",
        "else": "grant_basic_access"
      }
    },
    {
      "id": "grant_sandbox_access",
      "type": "transform",
      "config": {
        "operation": "map",
        "mapping": {
          "assignedCluster": "k8s-sandbox-cluster",
          "apiTokens": 50000
        }
      }
    },
    {
      "id": "grant_basic_access",
      "type": "transform",
      "config": {
        "operation": "map",
        "mapping": {
          "assignedCluster": "standard-cluster",
          "apiTokens": 5000
        }
      }
    },
    {
      "id": "ai_welcome_agent",
      "type": "ai",
      "config": {
        "provider": "openai",
        "prompt": "Generate onboarding message for {{steps.user_profile.name}} with cluster {{steps.grant_sandbox_access.assignedCluster ?? steps.grant_basic_access.assignedCluster}}",
        "mock_response": {
          "text": "Welcome Alex Mercer! Your sandbox environment is provisioned with 50,000 API tokens."
        }
      }
    }
  ]
}"""

with open('colab_workflow.json', 'w') as f:
    f.write(workflow_definition)

print("Workflow file created: colab_workflow.json")

In [ ]:
# [Cell 3] Run Workflow via CLI & Inspect State Checkpoints
!php bin/eidcloud-flow validate colab_workflow.json
!php bin/eidcloud-flow run colab_workflow.json --input='{"userId": 2048, "name": "Elena Rostova", "role": "developer"}' -v
!php bin/eidcloud-flow history --workflow=user_onboarding_demo